# Histórico de 7 dias

Cada linha da base final corresponde a um usuário e a uma data de previsão. O histórico contém os 7 dias de calendário anteriores. O alvo é o tempo que foi possível medir nos nove aplicativos no dia seguinte.

No Colab, envie `base_redes_sociais_modificada_v1.0.tsv` pela aba **Arquivos**. No Jupyter ou VS Code, abra o notebook com o TSV na mesma pasta de trabalho. Execute as células de cima para baixo.

In [1]:
from pathlib import Path
import pandas as pd

JANELA = 7
um_dia = pd.Timedelta(days=1)

## Leitura da base

Este é o TSV que saiu do primeiro notebook. A coluna de usuário já se chama `user_id`. A contagem abaixo mostra os eventos de cada aplicativo antes de remover duplicatas.

In [2]:
base = pd.read_csv('base_redes_sociais_modificada_v1.0.tsv', sep='\t')
base.head()

,user_id,session_id,timestamp,app_name,event_type
0,0,2,2018-01-16 06:26:10,Instagram,Opened
1,0,2,2018-01-16 06:26:21,Instagram,Closed
2,0,17,2018-01-16 13:53:12,YouTube,User Interaction
3,0,19,2018-01-16 14:45:47,Facebook,User Interaction
4,0,20,2018-01-16 15:20:40,Facebook,User Interaction


In [3]:
REDES_SOCIAIS = ['Facebook', 'Instagram', 'Twitter', 'Snapchat', 'Reddit',
                 'Pinterest', 'Discord', 'YouTube', 'Telegram']
base = base[base['app_name'].isin(REDES_SOCIAIS)].copy()
base['app_name'].value_counts()

app_name
Facebook     506209
Twitter      186339
Instagram    126008
Reddit       123069
Telegram      59370
YouTube       49540
Snapchat      15665
Pinterest      8383
Discord        7235
Name: count, dtype: int64

In [4]:
campos = ['user_id', 'session_id', 'timestamp', 'app_name', 'event_type']
tipos = ['Opened', 'Closed', 'Broken', 'User Interaction']
base['timestamp'] = pd.to_datetime(base['timestamp'], errors='coerce')
for coluna in ['user_id', 'session_id']:
    base[coluna] = pd.to_numeric(base[coluna], errors='coerce').astype('Int64')

validos = base[campos].notna().all(axis=1) & base['event_type'].isin(tipos)
print('Eventos nos aplicativos selecionados', len(base))
print('Eventos inválidos', (~validos).sum())
base = base.loc[validos, campos].copy()
print('Duplicatas exatas', base.duplicated().sum())
base = base.drop_duplicates().copy()

chaves = ['user_id', 'session_id', 'app_name']
base['ordem'] = base.index   # desempate pelo arquivo original
base = base.sort_values(chaves + ['timestamp', 'ordem'])
base['data'] = base['timestamp'].dt.normalize()
chave_dia = ['user_id', 'data']
print('Eventos após a limpeza', len(base))
print('Usuários', base['user_id'].nunique())

Eventos nos aplicativos selecionados 1081818
Eventos inválidos 0
Duplicatas exatas 611825
Eventos após a limpeza 469993
Usuários 265


## Medir os episódios

Dentro do mesmo usuário, sessão e aplicativo, um `Closed` encerra a última abertura válida. Uma segunda abertura reinicia a contagem. Um `Broken` interrompe o par. Uma interação pode ocorrer entre abertura e fechamento e, sozinha, apenas indica que houve atividade.

In [5]:
marcos = base.loc[base['event_type'] != 'User Interaction'].copy()
grupo = marcos.groupby(chaves, sort=False)
tipo_anterior = grupo['event_type'].shift()
tipo_seguinte = grupo['event_type'].shift(-1)
marcos['inicio'] = grupo['timestamp'].shift()

fechou = marcos['event_type'].eq('Closed') & tipo_anterior.eq('Opened')
abriu = marcos['event_type'].eq('Opened') & tipo_seguinte.eq('Closed')
episodios = marcos.loc[fechou, chaves + ['inicio', 'timestamp']].copy()
episodios = episodios.rename(columns={'timestamp': 'fim'})
if episodios.empty:
    raise ValueError('Nenhum par Opened e Closed foi encontrado no TSV.')

print('Pares de abertura e fechamento', len(episodios))
print('Pares com duração de zero segundo', episodios['inicio'].eq(episodios['fim']).sum())
episodios.head()

Pares de abertura e fechamento 201475
Pares com duração de zero segundo 8526


,user_id,session_id,app_name,inicio,fim
1,0,2,Instagram,2018-01-16 06:26:10,2018-01-16 06:26:21
9,0,41,Instagram,2018-01-17 05:10:45,2018-01-17 05:11:40
15,0,83,Instagram,2018-01-19 16:06:30,2018-01-19 16:07:27
17,0,84,Instagram,2018-01-19 16:15:13,2018-01-19 16:15:14
19,0,85,Instagram,2018-01-19 16:22:33,2018-01-19 16:22:33


Um episódio das 23h58 às 00h03 contribui com dois minutos no primeiro dia e três no segundo. A coluna `fim` guarda o fechamento original para verificar quando a duração passou a ser conhecida.

In [6]:
partes = []
pendentes = episodios.copy()
pendentes['data'] = pendentes['inicio'].dt.normalize()

while not pendentes.empty:
    meia_noite = pendentes['data'] + um_dia
    pendentes['inicio_trecho'] = pendentes['inicio'].clip(lower=pendentes['data'])
    pendentes['fim_trecho'] = pendentes['fim'].clip(upper=meia_noite)
    partes.append(pendentes[chave_dia + ['inicio_trecho', 'fim_trecho', 'fim']].copy())

    pendentes = pendentes.loc[pendentes['fim'] > meia_noite].copy()
    pendentes['data'] += um_dia

trechos = pd.concat(partes, ignore_index=True)
trechos.head()

,user_id,data,inicio_trecho,fim_trecho,fim
0,0,2018-01-16,2018-01-16 06:26:10,2018-01-16 06:26:21,2018-01-16 06:26:21
1,0,2018-01-17,2018-01-17 05:10:45,2018-01-17 05:11:40,2018-01-17 05:11:40
2,0,2018-01-19,2018-01-19 16:06:30,2018-01-19 16:07:27,2018-01-19 16:07:27
3,0,2018-01-19,2018-01-19 16:15:13,2018-01-19 16:15:14,2018-01-19 16:15:14
4,0,2018-01-19,2018-01-19 16:22:33,2018-01-19 16:22:33,2018-01-19 16:22:33


Ao somar o tempo diário, dois aplicativos abertos ao mesmo tempo contam aquele intervalo uma vez. Para a duração média por episódio, cada trecho mantém sua duração individual. Um par que abre e fecha no mesmo segundo conta como um episódio medido.

In [7]:
def resumir_dias(trechos):
    t = trechos.sort_values(chave_dia + ['inicio_trecho', 'fim_trecho']).copy()
    t['fim_acumulado'] = t.groupby(chave_dia)['fim_trecho'].cummax()
    fim_anterior = t.groupby(chave_dia)['fim_acumulado'].shift()

    # Só soma a parte que ainda não foi coberta por outro trecho.
    inicio = t['inicio_trecho'].where(fim_anterior.isna() | (t['inicio_trecho'] > fim_anterior),
                                       fim_anterior)
    t['min_novos'] = (t['fim_trecho'] - inicio).dt.total_seconds().clip(lower=0) / 60
    t['duracao_min'] = (t['fim_trecho'] - t['inicio_trecho']).dt.total_seconds() / 60
    return t.groupby(chave_dia, as_index=False).agg(
        minutos=('min_novos', 'sum'),
        episodios=('duracao_min', 'size'),
        media_episodio_min=('duracao_min', 'mean')
    )

## Conferir os dias parciais

Para cada interação, procuro o primeiro episódio do mesmo usuário, sessão e aplicativo cujo fechamento ocorreu naquele horário ou depois. Se a interação foi anterior à abertura desse episódio, ou se não há fechamento, sua duração fica desconhecida. As aberturas, fechamentos e quebras sem par também entram na contagem de pendências.

In [8]:
sem_par = marcos.loc[~(abriu | fechou), chave_dia]
interacoes = base.loc[base['event_type'].eq('User Interaction'),
                      chaves + ['data', 'timestamp']]
cobertura = pd.merge_asof(
    interacoes.sort_values('timestamp'), episodios.sort_values('fim'),
    left_on='timestamp', right_on='fim', by=chaves, direction='forward'
)
fora = cobertura['inicio'].isna() | (cobertura['timestamp'] < cobertura['inicio'])
sem_par = pd.concat([sem_par, cobertura.loc[fora, chave_dia]])
pendencias_dia = sem_par.groupby(chave_dia).size().rename('eventos_sem_par')

In [9]:
eventos_dia = base.groupby(chave_dia).size().reset_index(name='eventos')
diario = eventos_dia.merge(resumir_dias(trechos), on=chave_dia, how='outer')
diario = diario.merge(pendencias_dia, on=chave_dia, how='left')
contagens = ['eventos', 'episodios', 'eventos_sem_par']
diario[contagens] = diario[contagens].fillna(0).astype(int)

datas = diario.groupby('user_id')['data']
diario['borda'] = diario['data'].eq(datas.transform('min')) | diario['data'].eq(datas.transform('max'))
diario['situacao'] = 'Com tempo medido e sem pendência detectada'
diario.loc[diario['eventos_sem_par'] > 0, 'situacao'] = 'Com tempo medido e evento sem par'
diario.loc[diario['minutos'].isna(), 'situacao'] = 'Sem duração mensurável'

print('Dias de usuário no recorte', len(diario))
display(diario['situacao'].value_counts())
diario.head()

Dias de usuário no recorte 2435


situacao
Com tempo medido e evento sem par             2142
Com tempo medido e sem pendência detectada     175
Sem duração mensurável                         118
Name: count, dtype: int64

,user_id,data,eventos,minutos,episodios,media_episodio_min,eventos_sem_par,borda,situacao
0,0,2018-01-16,7,0.183333,1,0.183333,5,True,Com tempo medido e evento sem par
1,0,2018-01-17,3,0.916667,1,0.916667,1,False,Com tempo medido e evento sem par
2,0,2018-01-18,1,NaN,0,NaN,1,False,Sem duração mensurável
3,0,2018-01-19,14,1.133333,6,0.188889,2,False,Com tempo medido e evento sem par
4,0,2018-01-22,15,5.933333,7,0.847619,1,False,Com tempo medido e evento sem par


`borda` marca a primeira e a última data de cada usuário no TSV filtrado. Ambas podem participar. A contagem de eventos sem par sinaliza problemas visíveis na data do registro. Mesmo em um dia sem pendências detectadas, a cobertura de 24 horas é desconhecida.

Na tabela diária, `NaN` em `minutos` indica que nenhum intervalo pôde ser medido. Já `0.0` pode ocorrer quando os pares medidos começaram e terminaram no mesmo segundo. O arquivo contém apenas os nove aplicativos, então uma data inteira sem evidência fica ausente.

## Montar a janela de 7 dias

`min_d1` é o tempo do dia anterior ao alvo. `min_d7` é o tempo do dia mais antigo da janela. Cada ligação usa o mesmo usuário e a data exata. Um dia ausente quebra a sequência. Um dia com eventos e sem tempo medido pode entrar no histórico com `NaN`.

Para cada data-alvo, uma entrada só usa episódios cujo fechamento já havia ocorrido antes da meia-noite que inicia o alvo. O alvo usa os trechos medidos na própria data, inclusive quando um trecho é confirmado por um fechamento posterior.

In [10]:
janela = diario.loc[diario['minutos'].notna(),
                    ['user_id', 'data', 'minutos', 'eventos_sem_par', 'borda']].copy()
janela = janela.rename(columns={'data': 'data_alvo', 'minutos': 'alvo_min',
                                'eventos_sem_par': 'eventos_sem_par_alvo', 'borda': 'borda_alvo'})

for atraso in range(1, JANELA + 1):
    intervalo = pd.Timedelta(days=atraso)
    corte = trechos['data'] + intervalo
    disponiveis = trechos.loc[trechos['fim'] < corte]
    dia = eventos_dia.merge(resumir_dias(disponiveis), on=chave_dia, how='outer')
    dia['data_alvo'] = dia['data'] + intervalo
    dia[f'min_d{atraso}'] = dia['minutos']

    colunas = ['user_id', 'data_alvo', f'min_d{atraso}']
    if atraso == 1:
        dia['episodios_d1'] = dia['episodios'].fillna(0).astype(int)
        dia['media_episodio_d1_min'] = dia['media_episodio_min']
        colunas += ['episodios_d1', 'media_episodio_d1_min']

    janela = janela.merge(dia[colunas], on=['user_id', 'data_alvo'],
                          how='inner', validate='one_to_one')

In [11]:
lags = [f'min_d{i}' for i in range(1, JANELA + 1)]
janela['dias_com_tempo'] = janela[lags].notna().sum(axis=1)
print('Sequências com data-alvo mensurável', len(janela))
print('Sequências sem nenhum dia de histórico mensurável', janela['dias_com_tempo'].eq(0).sum())
janela = janela.loc[janela['dias_com_tempo'] > 0].sort_values(['user_id', 'data_alvo']).copy()
janela['inicio_historico'] = janela['data_alvo'] - pd.Timedelta(days=JANELA)
janela['fim_historico'] = janela['data_alvo'] - um_dia

Sequências com data-alvo mensurável 702
Sequências sem nenhum dia de histórico mensurável 0


## Resumir o histórico

Nesta janela, comparo os 3 dias mais recentes com os 4 anteriores. A tendência é a média do bloco recente menos a média do anterior. As médias e o desvio-padrão populacional usam os dias com minutos medidos. `dias_com_tempo` mostra quantos entraram na média da janela. Se um bloco inteiro ficar sem medida, a tendência fica em branco.

In [12]:
recentes = JANELA // 2
janela['media_janela_min'] = janela[lags].mean(axis=1)
janela['media_recente_min'] = janela[lags[:recentes]].mean(axis=1)
media_anterior = janela[lags[recentes:]].mean(axis=1)
janela['tendencia_min'] = janela['media_recente_min'] - media_anterior
janela['desvio_janela_min'] = janela[lags].std(axis=1, ddof=0)
janela['desvio_d1_min'] = janela['min_d1'] - janela['media_janela_min']
janela['alvo_horas'] = janela['alvo_min'] / 60

# O dia da semana do alvo já é conhecido no momento da previsão.
semana = ['seg', 'ter', 'qua', 'qui', 'sex', 'sab', 'dom']
colunas_semana = [f'dia_{nome}' for nome in semana]
for numero, nome in enumerate(colunas_semana):
    janela[nome] = janela['data_alvo'].dt.dayofweek.eq(numero).astype(int)

## Conferir uma linha e salvar

In [13]:
print('Janelas', len(janela))
print('Usuários representados', janela['user_id'].nunique())
print('Janelas com tempo medido em todos os dias do histórico', janela['dias_com_tempo'].eq(JANELA).sum())
print('Alvos com tempo medido e algum evento sem par', janela['eventos_sem_par_alvo'].gt(0).sum())
print('Alvos na borda do recorte', janela['borda_alvo'].sum())
janela[['user_id', 'inicio_historico', 'data_alvo'] + lags[::-1] + ['alvo_min']].head()

Janelas 702
Usuários representados 60
Janelas com tempo medido em todos os dias do histórico 623
Alvos com tempo medido e algum evento sem par 689
Alvos na borda do recorte 41


,user_id,inicio_historico,data_alvo,min_d7,min_d6,min_d5,min_d4,min_d3,min_d2,min_d1,alvo_min
0,5,2018-01-19,2018-01-26,31.833333,107.616667,272.133333,183.550000,42.050000,52.866667,121.333333,42.250000
1,5,2018-01-20,2018-01-27,107.616667,272.133333,183.550000,42.050000,52.866667,121.650000,42.250000,65.233333
2,5,2018-01-21,2018-01-28,272.133333,183.550000,42.050000,52.866667,121.650000,42.250000,65.233333,38.816667
3,5,2018-01-22,2018-01-29,183.550000,42.050000,52.866667,121.650000,42.250000,65.233333,38.816667,44.283333
4,5,2018-01-23,2018-01-30,42.050000,52.866667,121.650000,42.250000,65.233333,38.816667,44.283333,157.916667


In [14]:
if not janela.empty:
    exemplo = janela.iloc[0]
    print('Usuário', exemplo['user_id'])
    print('Data-alvo', exemplo['data_alvo'].date())
    print('Minutos medidos no alvo', round(exemplo['alvo_min'], 2))
    display(pd.DataFrame({
        'data_do_historico': pd.date_range(exemplo['inicio_historico'], periods=JANELA),
        'minutos_medidos': [exemplo[nome] for nome in lags[::-1]]
    }))

Usuário 5
Data-alvo 2018-01-26
Minutos medidos no alvo 42.25


,data_do_historico,minutos_medidos
0,2018-01-19,31.833333
1,2018-01-20,107.616667
2,2018-01-21,272.133333
3,2018-01-22,183.550000
4,2018-01-23,42.050000
5,2018-01-24,52.866667
6,2018-01-25,121.333333


Para retomar a modelagem depois, a configuração A reúne os indicadores do último dia. A configuração B acrescenta os resumos do histórico. O código do usuário serve para separar treino e teste por pessoa. As colunas de auditoria do alvo só são conhecidas depois da previsão.

Se usar Ridge com dias sem minutos, ajuste a imputação apenas com os dados de treino em cada divisão. Para inspecionar apenas as linhas com tempo em todos os dias do histórico, use `janela[janela['dias_com_tempo'] == JANELA]`. Esse filtro ainda aceita dias parcialmente medidos.

In [15]:
ultimo_dia = ['min_d1', 'episodios_d1', 'media_episodio_d1_min']
resumo = ['dias_com_tempo', 'media_janela_min', 'media_recente_min',
          'desvio_janela_min', 'tendencia_min', 'desvio_d1_min']
X_ultimo_dia = janela[ultimo_dia + colunas_semana]
X_historico = janela[ultimo_dia + resumo + colunas_semana]
y = janela['alvo_min']  # para trabalhar em horas, use janela['alvo_horas']

In [16]:
colunas = (['user_id', 'inicio_historico', 'fim_historico', 'data_alvo']
           + lags[::-1] + ultimo_dia[1:] + resumo + colunas_semana
           + ['alvo_min', 'alvo_horas', 'eventos_sem_par_alvo', 'borda_alvo'])

pasta = Path('resultados')
pasta.mkdir(exist_ok=True)
arquivo = pasta / f'janela_{JANELA}_dias.tsv'
janela[colunas].to_csv(arquivo, sep='\t', index=False, date_format='%Y-%m-%d')
print(f'{len(janela)} linhas salvas em {arquivo}')

702 linhas salvas em resultados/janela_7_dias.tsv


No Colab, o TSV fica em **Arquivos → resultados**. Use o menu ao lado do nome do arquivo para baixar.